# WikiArt Style Classification (ResNet50 - Third Notebook)

This is a fresh third notebook focused on improving performance with stronger training settings.

Why this new notebook?
- Earlier notebooks reached around 50-60% validation accuracy
- We want better learning and better generalization
- We use a stronger pretrained model (`ResNet50`) and train for more epochs

## 1. Project Overview

In this notebook we will:
- Automatically locate WikiArt CSV files and image folders
- Build a custom PyTorch Dataset
- Filter bad rows (missing/unreadable images)
- Train a pretrained ResNet50 model
- Track train and validation loss/accuracy each epoch
- Evaluate on optional test split and run one prediction example

## 2. Imports and Device Setup

This cell imports required libraries and selects `cuda` if available.

In [1]:
from pathlib import Path
import random
import warnings
import copy
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.simplefilter("ignore", Image.DecompressionBombWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 3. Dataset Explanation and Preview

WikiArt style data is stored as:
- image files inside style folders
- CSV rows in the format `relative_path,label` (no header)

The code below auto-detects the project root and finds train/validation CSVs.

In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datasets").exists() and (candidate / "README.md").exists():
            return candidate
    return start.resolve()

def find_split_csv(data_dir: Path, split: str) -> Path:
    split = split.lower()
    csvs = sorted(data_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    preferred = [p for p in csvs if split in p.stem.lower() and "style" in p.stem.lower()]
    if preferred:
        return preferred[0]

    fallback = [p for p in csvs if split in p.stem.lower()]
    if fallback:
        return fallback[0]

    raise FileNotFoundError(f"Could not find a '{split}' CSV in {data_dir}")

project_root = find_project_root(Path.cwd())
wikiart_dir = project_root / "datasets" / "Wikiart"

train_csv = find_split_csv(wikiart_dir, "train")
val_csv = find_split_csv(wikiart_dir, "val")

train_df = pd.read_csv(train_csv, header=None, names=["relative_path", "label"])
val_df = pd.read_csv(val_csv, header=None, names=["relative_path", "label"])

print(f"Project root: {project_root}")
print(f"WikiArt dir:  {wikiart_dir}")
print(f"Train CSV:    {train_csv.name}")
print(f"Val CSV:      {val_csv.name}")
print(f"Train rows:   {len(train_df):,}")
print(f"Val rows:     {len(val_df):,}")
display(train_df.head())

Project root: C:\Users\Thijs\Desktop\Ai Art Critic
WikiArt dir:  C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart
Train CSV:    style_train.csv
Val CSV:      style_val.csv
Train rows:   57,025
Val rows:     24,421


,relative_path,label
0,Impressionism/edgar-degas_landscape-on-the-orn...,12
1,Realism/camille-corot_mantes-cathedral.jpg,21
2,Abstract_Expressionism/gene-davis_untitled-197...,0
3,Symbolism/kuzma-petrov-vodkin_in-the-1920.jpg,24
4,Impressionism/maurice-prendergast_paris-boulev...,12


In [3]:
def extract_style_name(relative_path: str) -> str:
    return Path(relative_path).parts[0]

train_df["style_name"] = train_df["relative_path"].map(extract_style_name)
val_df["style_name"] = val_df["relative_path"].map(extract_style_name)

label_to_style = (
    train_df[["label", "style_name"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["style_name"]
    .to_dict()
)

num_classes = len(label_to_style)
print(f"Number of classes: {num_classes}")
print("First 10 mappings:")
for lbl, style in list(label_to_style.items())[:10]:
    print(f"{lbl:2d} -> {style}")

Number of classes: 27
First 10 mappings:
 0 -> Abstract_Expressionism
 1 -> Action_painting
 2 -> Analytical_Cubism
 3 -> Art_Nouveau_Modern
 4 -> Baroque
 5 -> Color_Field_Painting
 6 -> Contemporary_Realism
 7 -> Cubism
 8 -> Early_Renaissance
 9 -> Expressionism


## 4. Image Transformations with Augmentation

Training augmentations:
- random horizontal flip
- random rotation (+/-10 degrees)
- color jitter (brightness/contrast/saturation)
- random resized crop (slight zoom/crop)

Validation/test transforms are simpler and deterministic.

All transforms use ImageNet normalization because ResNet50 is pretrained on ImageNet.

In [4]:
image_size = 224
batch_size = 32  # choose 32-64 depending on GPU memory
num_workers = 0  # beginner-friendly default on Windows

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

## 5. Custom Dataset Class

This dataset reads one row from the CSV and returns:
- transformed image tensor
- numeric class label

In [5]:
class WikiArtStyleDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_root: Path, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_root / row["relative_path"]
        label = int(row["label"])

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label

## 6. DataLoaders (Train, Validation, Optional Test)

Because WikiArt here has train + validation CSV only, we optionally split 15% of validation as test.

We also filter out rows with missing or unreadable image files before training.

In [6]:
def is_readable_image(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError):
        return False

def filter_valid_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    full_paths = df["relative_path"].map(lambda p: image_root / p)
    valid_mask = full_paths.map(is_readable_image)

    kept = int(valid_mask.sum())
    removed = int((~valid_mask).sum())
    print(f"{split_name}: kept {kept:,}, removed {removed:,} bad rows")

    cleaned = df.loc[valid_mask].reset_index(drop=True)
    if cleaned.empty:
        raise RuntimeError(f"No valid rows left for split '{split_name}'.")
    return cleaned

train_df_clean = filter_valid_rows(train_df, wikiart_dir, "train")
val_df_clean = filter_valid_rows(val_df, wikiart_dir, "validation")

CREATE_TEST_SPLIT = True
TEST_FRACTION_FROM_VAL = 0.15  # 10-20% range

if CREATE_TEST_SPLIT and len(val_df_clean) > 20:
    test_df = val_df_clean.sample(frac=TEST_FRACTION_FROM_VAL, random_state=SEED)
    val_eval_df = val_df_clean.drop(test_df.index).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
else:
    val_eval_df = val_df_clean
    test_df = None

train_dataset = WikiArtStyleDataset(train_df_clean, wikiart_dir, transform=train_transform)
val_dataset = WikiArtStyleDataset(val_eval_df, wikiart_dir, transform=eval_transform)
test_dataset = (WikiArtStyleDataset(test_df, wikiart_dir, transform=eval_transform) if test_df is not None else None)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers) if test_dataset is not None else None

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")
if test_dataset is not None:
    print(f"Test samples:  {len(test_dataset):,} (split from validation)")
else:
    print("Test set: not created")

train: kept 57,023, removed 2 bad rows
validation: kept 24,421, removed 0 bad rows
Train samples: 57,023
Val samples:   20,758
Test samples:  3,663 (split from validation)


## 7. Load Pretrained ResNet50 and Fine-Tune

We load ImageNet-pretrained ResNet50 and replace the final fully connected layer with the number of WikiArt classes.

In [7]:
LEARNING_RATE = 3e-4
EPOCHS = 10  # requested 8-12 range

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model.fc)
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {batch_size}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Thijs/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


Linear(in_features=2048, out_features=27, bias=True)
Learning rate: 0.0003
Epochs: 10
Batch size: 32


## 8. Training Loop with Progress Printing

Each epoch reports:
- train loss and train accuracy
- validation loss and validation accuracy

We save the best model state based on validation accuracy.

In [8]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

history = []
best_val_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer=None)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

history_df = pd.DataFrame(history)
display(history_df)

Epoch 01/10 | train_loss=1.6073, train_acc=0.472 | val_loss=1.3705, val_acc=0.539
Epoch 02/10 | train_loss=1.2893, train_acc=0.563 | val_loss=1.3029, val_acc=0.566
Epoch 03/10 | train_loss=1.1479, train_acc=0.607 | val_loss=1.2284, val_acc=0.587
Epoch 04/10 | train_loss=1.0354, train_acc=0.642 | val_loss=1.1799, val_acc=0.607
Epoch 05/10 | train_loss=0.9352, train_acc=0.676 | val_loss=1.1756, val_acc=0.608
Epoch 06/10 | train_loss=0.8416, train_acc=0.706 | val_loss=1.2423, val_acc=0.597
Epoch 07/10 | train_loss=0.7561, train_acc=0.735 | val_loss=1.1995, val_acc=0.625
Epoch 08/10 | train_loss=0.6821, train_acc=0.759 | val_loss=1.2524, val_acc=0.617
Epoch 09/10 | train_loss=0.6099, train_acc=0.785 | val_loss=1.2231, val_acc=0.632
Epoch 10/10 | train_loss=0.5479, train_acc=0.808 | val_loss=1.2899, val_acc=0.626


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,1.607338,0.471669,1.370525,0.539310
1,2,1.289323,0.563281,1.302912,0.566384
2,3,1.147925,0.606720,1.228399,0.587388
3,4,1.035401,0.641653,1.179921,0.607043
4,5,0.935166,0.675517,1.175630,0.607958
5,6,0.841627,0.706119,1.242350,0.596926
6,7,0.756110,0.735054,1.199487,0.624627
7,8,0.682136,0.759097,1.252433,0.616630
8,9,0.609928,0.785473,1.223149,0.631853
9,10,0.547883,0.808463,1.289896,0.625638


## 9. Validation and Test Evaluation

We evaluate final accuracy on validation and (if available) on test split from validation.

In [9]:
if best_state is not None:
    model.load_state_dict(best_state)

def evaluate_accuracy(model, loader):
    model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    return total_correct / total_samples

final_val_acc = evaluate_accuracy(model, val_loader)
print(f"Final validation accuracy: {final_val_acc:.3f}")

if test_loader is not None:
    final_test_acc = evaluate_accuracy(model, test_loader)
    print(f"Final test accuracy:       {final_test_acc:.3f}")

Final validation accuracy: 0.632
Final test accuracy:       0.632


## 10. Multi-Image Prediction Example

This example predicts style for **5 images** and prints true vs predicted labels.

In [10]:
def predict_image_style(model, image_path: Path, transform, label_to_style_map):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        pred_label = int(outputs.argmax(dim=1).item())

    pred_style = label_to_style_map.get(pred_label, f"Unknown label {pred_label}")
    return pred_label, pred_style

# Pick 5 examples from test split if available, otherwise from validation
if test_df is not None and len(test_df) > 0:
    sample_df = test_df.sample(n=min(5, len(test_df)), random_state=SEED).reset_index(drop=True)
else:
    sample_df = val_eval_df.sample(n=min(5, len(val_eval_df)), random_state=SEED).reset_index(drop=True)

print("Prediction examples (5 images):")
for i, row in sample_df.iterrows():
    sample_relative_path = row["relative_path"]
    true_label = int(row["label"])
    sample_image_path = wikiart_dir / sample_relative_path

    pred_label, pred_style = predict_image_style(model, sample_image_path, eval_transform, label_to_style)
    true_style = label_to_style.get(true_label, f"Unknown label {true_label}")

    print(f"\nExample {i + 1}")
    print(f"Image:           {sample_relative_path}")
    print(f"True style:      {true_style} ({true_label})")
    print(f"Predicted style: {pred_style} ({pred_label})")

Prediction examples (5 images):

Example 1
Image:           Minimalism/piero-manzoni_achrome-1960-2.jpg
True style:      Minimalism (14)
Predicted style: Minimalism (14)

Example 2
Image:           Realism/valentin-serov_winter-in-abramtsevo-1886-1.jpg
True style:      Realism (21)
Predicted style: Realism (21)

Example 3
Image:           Pop_Art/patrick-caulfield_cream-glazed-pot-1979.jpg
True style:      Pop_Art (19)
Predicted style: Ukiyo_e (26)

Example 4
Image:           Impressionism/james-mcneill-whistler_rose-and-brown-la-cigale.jpg
True style:      Impressionism (12)
Predicted style: Symbolism (24)

Example 5
Image:           Symbolism/william-blake_the-parable-of-the-wise-and-foolish-virgins-1822.jpg
True style:      Symbolism (24)
Predicted style: Symbolism (24)


## 11. Next Steps (Web App Integration)

To use this in your app later:
1. Save best model weights (`torch.save`)
2. Create backend endpoint for image upload
3. Preprocess uploaded image with the same eval transform
4. Run inference and return predicted style
5. Display prediction + confidence in frontend

For even better performance later, you can add: learning-rate scheduler, early stopping, and per-class metrics/confusion matrix.